<a href="https://colab.research.google.com/github/jhughes7386/cosc-650-applied-llm-systems/blob/week-05/week-05/week5_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 (starter): RAG Pipeline with Retrieval Evaluation

Retrieval runs fully local and free; only generation needs a key (set `GEMINI_API_KEY`, else it is skipped with a notice). Cells marked **TODO (you)** are yours. Dependencies: `sentence-transformers`, `faiss-cpu`. For generation: `pip install openai`.

In [ ]:
import os, re, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# TODO (you): replace with your own technical-doc corpus. This placeholder is one short doc.
DOC = '''## Networking\nThe Orbit server listens on TCP port 7700 by default. Gossip runs on 7711.\n\n## Storage\nOrbit persists data with an LSM-tree storage engine.\n\n## Replication\nEach key is replicated with a default replication factor of 3. Default consistency is quorum.\n\n## Limits\nThe maximum value size is 16 MB. A batch may contain at most 1000 operations.\n\n## Security\nClients authenticate with API tokens. Traffic is encrypted with TLS.'''
print('corpus chars:', len(DOC))

In [ ]:
def chunk_fixed(text, size, overlap):
    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out
def chunk_paragraph(text):
    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]

configs = {'small (120/20)': chunk_fixed(DOC,120,20), 'large (320/40)': chunk_fixed(DOC,320,40), 'by-paragraph': chunk_paragraph(DOC)}
for n, c in configs.items():
    print(f'{n:16s} -> {len(c)} chunks')

def build(chunks):
    e = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx
def retrieve(idx, chunks, q, k=3):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    # FAISS uses -1 for missing results when fewer than k chunks exist.
    return [chunks[i] for i in idx.search(qe, k)[1][0] if i >= 0]

## Part 2: Evaluate retrieval (compare chunking configs)
A query is answered only if the retrieved context contains the answer fact. This is the model-free half of answer quality.

In [ ]:
# TODO (you): your own queries and the fact each answer must contain.
qa = [('what port does orbit use','7700'), ('which storage engine','LSM'),
      ('default replication factor','replication factor of 3'), ('maximum value size','16 MB'),
      ('how do clients authenticate','API token'), ('default consistency','quorum')]
indices = {n: build(c) for n, c in configs.items()}
# This example checks whether retrieved context contains the answer.
# It does not calculate the chunk-level precision and recall required below.
for n, c in configs.items():
    hits = sum(1 for q, fact in qa if fact.lower() in ' '.join(retrieve(indices[n], c, q)).lower())
    print(f'{n:16s} answer-recall@3: {hits}/{len(qa)} = {hits/len(qa):.2f}')
# TODO (you): compare at least two embedding models OR at least three chunking
# configurations on at least ten queries. Label relevant chunks in advance,
# report chunk-level precision and recall, and explain where the choices disagree.

## Part 3: Generate (needs a key)
Build a grounded prompt from the retrieved chunks and answer with Gemini.

In [ ]:
def gemini_chat(messages, model='gemini-2.5-flash', **kw):
    if not os.environ.get('GEMINI_API_KEY'):
        return None
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

def answer(q, config='by-paragraph'):
    ctx = retrieve(indices[config], configs[config], q, k=3)
    prompt = 'Answer only from the context.\nContext:\n- ' + '\n- '.join(ctx) + f'\nQuestion: {q}\nAnswer:'
    out = gemini_chat([{'role':'user','content':prompt}])
    return out if out is not None else '[API-BLOCKED] set GEMINI_API_KEY to generate; grounded prompt was built'

print(answer('what port does orbit use'))

## Part 4 and 5: failure, submit
**TODO (you):** show a query where retrieval surfaces the wrong chunk or a chunk config splits a fact, and explain the mitigation. Then open a pull request with a result summary and review a classmate's Week 5 PR (the term's formal peer review). Use the rubric attached to the Canvas assignment for grading criteria and point values.